In [1]:
!pip install transformers datasets scikit-learn pandas torch


In [2]:
import pandas as pd
import random

sales = [
"I want to upgrade my plan",
"Interested in enterprise pricing",
"Can you share product pricing?"
]

support = [
"I need help resetting password",
"Unable to login to my account",
"App keeps crashing"
]

complaint = [
"My order arrived late",
"Very disappointed with service",
"Refund not processed"
]

feedback = [
"Great product experience",
"Loved the new update",
"Service was excellent"
]

data = []

for _ in range(250):
    data.append([random.choice(sales),"sales"])
    data.append([random.choice(support),"support"])
    data.append([random.choice(complaint),"complaint"])
    data.append([random.choice(feedback),"feedback"])

df = pd.DataFrame(data,columns=["text","label"])

df.to_csv("dataset.csv",index=False)

df.head()

,text,label
0,Can you share product pricing?,sales
1,I need help resetting password,support
2,My order arrived late,complaint
3,Loved the new update,feedback
4,Interested in enterprise pricing,sales


In [3]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label_encoded"] = encoder.fit_transform(df["label"])

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
model_name,
num_labels=4
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
texts = df["text"].tolist()
labels = df["label_encoded"].tolist()

encodings = tokenizer(
texts,
truncation=True,
padding=True
)

In [6]:
import torch

class Dataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = Dataset(encodings, labels)

In [7]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
output_dir="./results",
num_train_epochs=2,
per_device_train_batch_size=8
)

trainer = Trainer(
model=model,
args=training_args,
train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.10092964172363281, metrics={'train_runtime': 372.918, 'train_samples_per_second': 5.363, 'train_steps_per_second': 0.67, 'total_flos': 9250164144000.0, 'train_loss': 0.10092964172363281, 'epoch': 2.0})

In [8]:
def predict(text):

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model(**inputs)

    pred = outputs.logits.argmax().item()

    return encoder.inverse_transform([pred])[0]

predict("Customer is unhappy with delayed delivery")

'complaint'

In [9]:
model.save_pretrained("bert_classifier")
tokenizer.save_pretrained("bert_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_classifier/tokenizer_config.json', 'bert_classifier/tokenizer.json')

In [10]:
!mkdir bert-nlp-classifier

In [11]:
!cp dataset.csv bert-nlp-classifier/

In [12]:
%%writefile bert-nlp-classifier/requirements.txt
transformers
torch
pandas
scikit-learn

Writing bert-nlp-classifier/requirements.txt


In [13]:
%%writefile bert-nlp-classifier/README.md
# BERT NLP Text Classifier

Transformer-based NLP classifier using BERT.

Features
- Business message classification
- Transformer embeddings
- Model training pipeline

Tech Stack
Python
Transformers
PyTorch

Writing bert-nlp-classifier/README.md


In [14]:
!zip -r bert-nlp-classifier.zip bert-nlp-classifier

  adding: bert-nlp-classifier/ (stored 0%)
  adding: bert-nlp-classifier/README.md (deflated 31%)
  adding: bert-nlp-classifier/dataset.csv (deflated 97%)
  adding: bert-nlp-classifier/requirements.txt (stored 0%)


In [15]:
%%writefile bert-nlp-classifier/train.py
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("dataset.csv")

# Encode labels
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["label"])

texts = df["text"].tolist()
labels = df["label_encoded"].tolist()

# Load BERT
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

# Tokenize
encodings = tokenizer(texts, truncation=True, padding=True)

# Dataset class
class Dataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = Dataset(encodings, labels)

# Train
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Writing bert-nlp-classifier/train.py


In [16]:
!ls bert-nlp-classifier

dataset.csv  README.md	requirements.txt  train.py


In [17]:
!zip -r bert-nlp-classifier.zip bert-nlp-classifier

updating: bert-nlp-classifier/ (stored 0%)
updating: bert-nlp-classifier/README.md (deflated 31%)
updating: bert-nlp-classifier/dataset.csv (deflated 97%)
updating: bert-nlp-classifier/requirements.txt (stored 0%)
  adding: bert-nlp-classifier/train.py (deflated 56%)


In [18]:
!pip install nbstripout
!nbstripout bert_text_classifier.ipynb

Could not strip 'bert_text_classifier.ipynb': file not found


In [19]:
!ls

bert_classifier      bert-nlp-classifier.zip  results
bert-nlp-classifier  dataset.csv	      sample_data
